# 5 · O aumento dos juros foi proposital para beneficiar grupos?

**O que este notebook FAZ com dados:** apresenta o conjunto de **fatos
verificáveis** que qualquer hipótese de "propositalidade" precisa explicar.

**O que este notebook NÃO faz nesta fase:** afirmar que houve ou não intenção —
isso exige documentos (atas, manifestos, posicionamentos; fase posterior de
*posições/documentos*).

**Fatos mostrados aqui:**
1. trajetória do juro real ex ante por mandato;
2. dispêndio crescente com juros nominais e em % do PIB;
3. papel recorrente dos swaps (custo sistemático do jogo cambial do BCB);
4. o prêmio sobre a regra de Taylor (N1) como escolha de política, não automático.

In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import (
    estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA, CORES_DETENTOR,
)
from src.processa import indicadores as ind

estilo()
print("diretório de dados:", p / "dados" / "processado")


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado


In [2]:
mm = ind.selic_real_mensal()
mandatos = [
    ("Lula 1-2 (03-10)", "2003-01-01", "2010-12-31"),
    ("Dilma/Temer (11-18)", "2011-01-01", "2018-12-31"),
    ("Bolsonaro (19-22)", "2019-01-01", "2022-12-31"),
    ("Lula 3 (23-26)", "2023-01-01", "2026-12-31"),
]
linhas = []
for nome, ini, fim in mandatos:
    rec = mm.loc[ini: fim, "juro_real_ex_ante"].dropna()
    s = mm.loc[ini: fim, "selic_meta"].dropna()
    linhas.append({
        "mandato": nome,
        "juro real ex ante médio": rec.mean(),
        "máximo": rec.max(),
        "Selic média": s.mean(),
    })
tab = pd.DataFrame(linhas).set_index("mandato")
print(tab.round(1).to_string())

                     juro real ex ante médio  máximo  Selic média
mandato                                                          
Lula 1-2 (03-10)                        9.60   19.00        14.80
Dilma/Temer (11-18)                     4.90    9.20        10.50
Bolsonaro (19-22)                       2.10    8.90         6.60
Lula 3 (23-26)                          8.70   11.00        13.20


In [3]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8),
                               gridspec_kw={"height_ratios": [2, 1.1]})
cores = [VERMELHO, AZUL, CINZA, VERDE]
r = mm["juro_real_ex_ante"].dropna()

# topo: trajetória mensal — mostra o que a média por mandato esconde
for (nome, ini, fim), cor in zip(mandatos, cores):
    a, b = pd.Timestamp(ini), pd.Timestamp(fim)
    rec = mm.loc[ini: fim, "juro_real_ex_ante"].dropna()
    ax1.axvspan(a, b, color=cor, alpha=0.08)
    ax1.hlines(rec.mean(), a, min(b, r.index.max()), color=cor,
               lw=2.5, ls="--", zorder=4)
    ax1.text(a + (b - a) / 2, rec.mean() + 0.4, f"média {rec.mean():.1f}%",
             ha="center", va="bottom", fontsize=9, fontweight="bold", color=cor)

ax1.plot(r.index, r.values, color="black", lw=1.1, zorder=3)
ax1.axhline(0, color="gray", lw=0.8, zorder=1)
ax1.set_ylim(-6, 12.5)

# extremos que a média esconde (dentro dos eixos, sem colidir com os rótulos)
ax1.annotate("−2,3% em mar/21: fundo da pandemia",
             xy=(pd.Timestamp("2021-03-31"), float(r.asof("2021-03-31"))),
             xytext=(52, -8), textcoords="offset points",
             ha="left", va="center", fontsize=9,
             arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))
ax1.annotate("8,9% em set/22: o aperto já estava no topo",
             xy=(pd.Timestamp("2022-09-30"), float(r.asof("2022-09-30"))),
             xytext=(-14, 22), textcoords="offset points",
             ha="right", va="bottom", fontsize=9,
             arrowprops=dict(arrowstyle="->", color="gray", lw=0.8))

ax1.set_ylabel("% a.a.")
ax1.set_title("Juro real ex ante mensal por mandato — o tracejado é a média de cada mandato")
ax1.margins(x=0.01)

# base: as barras originais (média por mandato)
tab["juro real ex ante médio"].plot(kind="bar", ax=ax2, color=cores)
for i, valor in enumerate(tab["juro real ex ante médio"]):
    ax2.text(i, valor + 0.1, f"{valor:.1f}%", ha="center", fontsize=10,
             fontweight="bold")
ax2.set_ylabel("% a.a. (média do mandato)")
ax2.set_title("Média do mandato — cada barra cobre um ciclo completo de alta e baixa")
ax2.set_ylim(0, 11)
salvar(fig, "05_mandatos")

fig -> viz/figs/05_mandatos.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/05_mandatos.png')

In [4]:
v = ind.juros_anuais()
v = v[v.ano >= 2002]
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(v.ano, v.juros_mi / 1000, color=VERMELHO, label="juros nominais anuais (eixo esq.)")
ax2 = ax.twinx()
ax2.plot(v.ano, v.juros_pib_pct, color="k", marker="o", ms=4, label="juros % PIB (eixo dir.)")
ax.set_ylabel("R$ bilhões")
ax2.set_ylabel("% do PIB")
j02 = f"{v.loc[v.ano == 2002, 'juros_mi'].iloc[0] / 1000:,.0f}".replace(",", ".")
j25 = f"{v.loc[v.ano == 2025, 'juros_mi'].iloc[0] / 1000:,.0f}".replace(",", ".")
ax.set_title(f"Dispêndio com juros nominais: R\\$ {j02} bi (2002) → R\\$ {j25} bi (2025)")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=9)
salvar(fig, "05_juros_nominais")

fig -> viz/figs/05_juros_nominais.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/05_juros_nominais.png')

In [5]:
fig, ax = plt.subplots(figsize=(11, 4.5))
sw = v[(v.ano >= 2008) & (v.ano < 2026)]
cores = [VERDE if x < 0 else VERMELHO for x in sw.swap_pp_pib]
ax.bar(sw.ano, sw.swap_pp_pib, color=cores)
ax.axhline(0, color="k", lw=1)
ax.set_ylabel("p.p. do PIB (+ aumenta juros / − reduz)")
ax.set_title("Resultado caixa dos swaps do BCB: contribuição para os juros por ano")
salvar(fig, "05_swaps_pp")

fig -> viz/figs/05_swaps_pp.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/05_swaps_pp.png')

**Achados verificáveis (P5).**

- A régua de juro real **alternou de forma dramática e reversível entre
  mandatos**: ~9,6% (média ex ante em Lula 1–2) → 4,9% (Dilma/Temer) →
  2,1% (2019–22) → 8,7% (Lula 3). Níveis do mesmo sistema econômico,
  em intervalo de meia década, variam em ~6 p.p. — e o reajuste atual
  (2 → 9) foi o mais rápido do histórico. Isso **não prova intenção**;
  mostra apenas que o patamar é *escolha de política*, não fatalidade
  técnica: um país com juro real ex ante **−0,3% em 2021** (média 2019–22 de
  2,1%) praticou **9,6% em 2025** sem mudança de estrutura produtiva.
- O dispêndio nominal com juros cresceu ~8,9× entre 2002 e 2025 (R$ 113 bi →
  R$ 1.008 bi); os saltos de % do PIB ocorrem em **2015–16 e 2024–25**,
  ambos períodos de juro real
  *acima* do que a regra simples pediria (N1).
- Os swaps geraram custo em 8 dos 18 anos (44%, 2008–25) — um "imposto
  cambial" recorrente na conta de juros, contratado pelo BCB/Tesouro em
  diferentes gestões.

**Fase de documentos (próximo bloco, acordo pendente):** manifestos públicos
das entidades do mercado financeiro, ofícios e discursos institucionais do
BCB/Fazenda, atas e polêmicas registradas, composições de comitês — para
mapear **quem defendeu o patamar** e **quem tratou isso como prioridade**.
Esse material **não está nos datasets** deste projeto e exige coleta manual
e leitura qualificada.

- **Cuidado com as médias do gráfico 05:** cada barra cobre um ciclo
  completo. Dilma/Temer foi de 1,4% (jan/13) a 9,2% (out/16, com Selic em
  14,25%) e só depois caiu; Bolsonaro foi de −2,3% (mar/21) a 8,9%
  (set/22). O aperto atual começou em **março de 2021** e já estava
  no patamar de hoje em **dez/2022 (8,4%)** — a troca de nível não
  coincide com a troca de mandato.
